# Views, Identity, and Safe Migration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_OER/blob/main/course_materials/notebooks/03_views_identity_migration.ipynb)
[View this notebook on GitHub](https://github.com/lolusername/CST4714_OER/blob/main/course_materials/notebooks/03_views_identity_migration.ipynb)

Day 1 practices views and identity gaps. Day 2 rehearses a safe schema change. Keep the same practice database for both days. This walkthrough supports Chapter 4 and the OER Week 4 labs. Use the SQL submission filenames specified in those labs, not an additional notebook submission.

Use a **Python 3 / CPU** Colab runtime. Run cells individually from the
top; pause at **Your work**. The database is temporary: download your
SQL before disconnecting or running cleanup. No cloud database account,
password, GPU, or notebook submission is required.

## Meet Metro Support Before Writing Queries

Metro Support is a fictional public-service help desk. All people and
records here are synthetic. A table's **grain** is what one row means.

| Table | One row means | Rows | Important columns |
|---|---|---:|---|
| `users` | one resident or staff member | 8 | `user_id`, `display_name` |
| `tickets` | one support request | 12 | `ticket_id`, `category`, `priority`, `status`, `assignee_id` |
| `ticket_events` | one recorded event on a ticket | 21 | `event_id`, `ticket_id`, `event_type`, `event_at` |

A ticket has one requester and an optional assignee, both referring to
`users.user_id`. An event refers to `tickets.ticket_id`. One ticket can
therefore have several event rows; an event is **not** another ticket.

**Active** means status `new`, `open`, or `in_progress`: 7 of the 12
tickets. Tickets **1004 and 1009** are active but have a `NULL` assignee
(not assigned yet). Do not drop them from a report. Every ticket in the
starting fixture has at least one event; future tickets might not.

The setup below contains the exact published
[8-user / 12-ticket / 21-event SQL fixture](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_03/materials/datasets/metro_support/postgres_setup.sql).
It is embedded here, so running the notebook does **not** fetch data
from GitHub. Read the `CREATE TABLE` definitions and a few `INSERT`
rows when you need to see where a result came from.

## Setup: A Real, Disposable PostgreSQL Database

In Colab, the next cell installs PostgreSQL with `apt-get` and starts
its local service. `sudo -u postgres` runs database commands as the
service's operating-system user, without a password. Package
installation needs internet access; the dataset does not.

**Local Jupyter only:** use an already-running disposable PostgreSQL
server, put its `bin` directory on `PATH`, and explicitly set `PGHOST`
to its Unix socket directory before running this cell. Set `PGPORT`
and `PGUSER` too if they differ from your local defaults. Your role
needs permission to create databases. Remote hosts are refused.
This notebook does not install, start, or stop a local computer's server.

The next two cells are Python setup, not SQL to submit. A unique database
name keeps the fixture's schema reset away from existing databases.
Do not replace that generated name with one of your own databases.

In [ ]:
import getpass
import os
from pathlib import Path
import subprocess
from uuid import uuid4

# Local Jupyter only: uncomment and supply your actual socket directory.
# os.environ["PGHOST"] = "/path/to/your/postgresql/socket"
# os.environ["PGPORT"] = "5432"

if "PRACTICE_DB" in globals():
    raise RuntimeError("Keep this database for both days, or run cleanup first.")
if os.environ.get("PGHOSTADDR") or os.environ.get("PGSERVICE"):
    raise RuntimeError("Unset PGHOSTADDR and PGSERVICE; use an explicit local socket.")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "postgresql", "postgresql-client"], check=True)
    subprocess.run(["service", "postgresql", "start"], check=True)
    COLAB_SERVICE_STARTED = True

PGHOST = "/var/run/postgresql" if IN_COLAB else os.environ.get("PGHOST", "")
if not Path(PGHOST).is_absolute() or "," in PGHOST or not Path(PGHOST).is_dir():
    raise RuntimeError("Set PGHOST to one existing local Unix socket directory.")
PGPORT = "5432" if IN_COLAB else os.environ.get("PGPORT", "5432")
PGUSER = "postgres" if IN_COLAB else os.environ.get("PGUSER", getpass.getuser())
PG_PREFIX = ["sudo", "-u", "postgres"] if IN_COLAB else []
PG_ARGS = ["--host", PGHOST, "--port", PGPORT, "--username", PGUSER, "--no-password"]
subprocess.run(PG_PREFIX + ["pg_isready"] + PG_ARGS[:-1] + ["--dbname", "postgres"], check=True)
print("Local socket:", PGHOST, "Port:", PGPORT)

In [ ]:
# template0 supplies an empty database, not a copy of your course work.
if "PRACTICE_DB" in globals():
    raise RuntimeError("Database already created. Do not reset it between days.")
run_id = uuid4().hex
new_database = "cst4714_week03_" + run_id
subprocess.run(
    PG_PREFIX + ["createdb"] + PG_ARGS
    + ["--maintenance-db=postgres", "--template=template0", new_database],
    check=True,
)
PRACTICE_DB = new_database  # Remember only the database this run created.
print("This notebook owns:", PRACTICE_DB)

### One Small SQL Display Helper

`run_sql` sends the text between triple quotes to PostgreSQL's `psql`
program and prints its table output. Everything inside those quotes
is ordinary SQL. `-X` ignores personal psql settings; `ON_ERROR_STOP`
stops at the first error. `NULL` is printed explicitly, not as a blank.

Each call opens a **new connection**. Put `BEGIN`, every statement in
the transaction, and its `COMMIT` or `ROLLBACK` in **one call**. Do not
split a transaction or a temporary table across notebook cells.
Outside an explicit transaction, each statement commits separately.
On an error, the connection closes and any uncommitted transaction
rolls back; earlier committed work stays. Fix the first error and
rerun the complete intended block, not just its last line.

In [ ]:
def run_sql(sql_text):
    result = subprocess.run(
        PG_PREFIX + ["psql"] + PG_ARGS
        + ["-X", "--dbname", PRACTICE_DB, "--set=ON_ERROR_STOP=on",
           "--set=VERBOSITY=verbose", "--pset=null=NULL", "--pset=pager=off"],
        input=sql_text, text=True, capture_output=True,
    )
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(result.stderr.strip())

### Load the Fixture Once

The published script starts with `DROP SCHEMA ... CASCADE`. Here it
runs **only in the new practice database** just printed above. The
guard prevents accidentally rerunning it over your work in this
notebook. Do not copy this reset into a real database or your submission.
The final result must show **8 users, 12 tickets, and 21 events**.

In [ ]:
if globals().get("FIXTURE_LOADED", False):
    raise RuntimeError("Fixture already loaded. Keep your work; do not reset between days.")
assert PRACTICE_DB == "cst4714_week03_" + run_id
setup_sql = """-- Metro Support PostgreSQL setup
-- Run only in a course or personal practice database. This resets the
-- metro_support schema so the dataset is reproducible.

DROP SCHEMA IF EXISTS metro_support CASCADE;
CREATE SCHEMA metro_support;
SET search_path TO metro_support, public;

CREATE TABLE users (
    user_id integer PRIMARY KEY,
    display_name text NOT NULL,
    email text NOT NULL UNIQUE,
    role text NOT NULL,
    neighborhood text NOT NULL,
    created_at timestamptz NOT NULL
);

CREATE TABLE tickets (
    ticket_id integer PRIMARY KEY,
    requester_id integer NOT NULL REFERENCES users(user_id),
    assignee_id integer REFERENCES users(user_id),
    category text NOT NULL,
    priority text NOT NULL,
    status text NOT NULL,
    subject text NOT NULL,
    opened_at timestamptz NOT NULL,
    closed_at timestamptz,
    CHECK (closed_at IS NULL OR closed_at >= opened_at)
);

CREATE TABLE ticket_events (
    event_id integer PRIMARY KEY,
    ticket_id integer NOT NULL REFERENCES tickets(ticket_id),
    actor_id integer NOT NULL REFERENCES users(user_id),
    event_type text NOT NULL,
    old_status text,
    new_status text,
    note text,
    event_at timestamptz NOT NULL
);

INSERT INTO users
    (user_id, display_name, email, role, neighborhood, created_at)
VALUES
    (101, 'Maya Chen', 'maya.chen@example.test', 'resident', 'Harbor', '2026-01-08T14:20:00Z'),
    (102, 'Luis Rivera', 'luis.rivera@example.test', 'resident', 'Northside', '2026-01-10T09:15:00Z'),
    (103, 'Amina Yusuf', 'amina.yusuf@example.test', 'resident', 'Central', '2026-01-12T18:05:00Z'),
    (104, 'Jordan Bell', 'jordan.bell@example.test', 'resident', 'Harbor', '2026-01-18T11:40:00Z'),
    (201, 'Priya Shah', 'priya.shah@example.test', 'agent', 'Central', '2025-11-03T13:00:00Z'),
    (202, 'Noah Williams', 'noah.williams@example.test', 'agent', 'Northside', '2025-11-05T13:00:00Z'),
    (203, 'Elena Garcia', 'elena.garcia@example.test', 'supervisor', 'Central', '2025-09-14T13:00:00Z'),
    (204, 'Sam Okafor', 'sam.okafor@example.test', 'analyst', 'Harbor', '2025-12-01T13:00:00Z');

INSERT INTO tickets
    (ticket_id, requester_id, assignee_id, category, priority, status, subject, opened_at, closed_at)
VALUES
    (1001, 101, 201, 'streetlight', 'high', 'open', 'Streetlight dark near bus stop', '2026-02-01T23:10:00Z', NULL),
    (1002, 102, 202, 'sanitation', 'medium', 'in_progress', 'Missed recycling pickup', '2026-02-02T15:45:00Z', NULL),
    (1003, 103, 201, 'water', 'urgent', 'resolved', 'Low water pressure', '2026-02-03T12:05:00Z', '2026-02-03T19:40:00Z'),
    (1004, 104, NULL, 'parks', 'low', 'new', 'Broken bench slat', '2026-02-04T17:20:00Z', NULL),
    (1005, 101, 202, 'sanitation', 'high', 'resolved', 'Overflowing corner bin', '2026-02-05T14:00:00Z', '2026-02-05T20:15:00Z'),
    (1006, 102, 201, 'streetlight', 'medium', 'in_progress', 'Flickering lamp outside library', '2026-02-06T01:30:00Z', NULL),
    (1007, 103, 202, 'parks', 'medium', 'open', 'Playground gate will not latch', '2026-02-07T16:10:00Z', NULL),
    (1008, 104, 201, 'water', 'high', 'resolved', 'Hydrant leaking slowly', '2026-02-08T10:25:00Z', '2026-02-09T09:05:00Z'),
    (1009, 101, NULL, 'transportation', 'medium', 'new', 'Bus shelter panel cracked', '2026-02-09T22:15:00Z', NULL),
    (1010, 102, 202, 'sanitation', 'low', 'closed', 'Replacement bin request', '2026-02-10T13:50:00Z', '2026-02-12T16:30:00Z'),
    (1011, 103, 201, 'transportation', 'high', 'open', 'Crosswalk signal delayed', '2026-02-11T08:35:00Z', NULL),
    (1012, 104, 202, 'streetlight', 'low', 'resolved', 'Lamp stays on during daytime', '2026-02-12T14:45:00Z', '2026-02-14T18:10:00Z');

INSERT INTO ticket_events
    (event_id, ticket_id, actor_id, event_type, old_status, new_status, note, event_at)
VALUES
    (5001, 1001, 101, 'created', NULL, 'open', 'Reported through mobile form', '2026-02-01T23:10:00Z'),
    (5002, 1001, 201, 'assigned', 'open', 'open', 'Electrical crew notified', '2026-02-02T14:05:00Z'),
    (5003, 1002, 102, 'created', NULL, 'open', 'Pickup was scheduled for Monday', '2026-02-02T15:45:00Z'),
    (5004, 1002, 202, 'status_changed', 'open', 'in_progress', 'Route supervisor checking vehicle log', '2026-02-03T13:30:00Z'),
    (5005, 1003, 103, 'created', NULL, 'open', 'Pressure lower on two floors', '2026-02-03T12:05:00Z'),
    (5006, 1003, 201, 'status_changed', 'open', 'in_progress', 'Crew dispatched', '2026-02-03T14:25:00Z'),
    (5007, 1003, 201, 'status_changed', 'in_progress', 'resolved', 'Valve adjustment restored pressure', '2026-02-03T19:40:00Z'),
    (5008, 1004, 104, 'created', NULL, 'new', 'Photo attached in original report', '2026-02-04T17:20:00Z'),
    (5009, 1005, 101, 'created', NULL, 'open', 'Bin blocks part of sidewalk', '2026-02-05T14:00:00Z'),
    (5010, 1005, 202, 'status_changed', 'open', 'resolved', 'Extra collection completed', '2026-02-05T20:15:00Z'),
    (5011, 1006, 102, 'created', NULL, 'open', 'Flicker repeats every few seconds', '2026-02-06T01:30:00Z'),
    (5012, 1006, 201, 'status_changed', 'open', 'in_progress', 'Ballast inspection scheduled', '2026-02-06T15:10:00Z'),
    (5013, 1007, 103, 'created', NULL, 'open', 'Gate opens toward play area', '2026-02-07T16:10:00Z'),
    (5014, 1008, 104, 'created', NULL, 'open', 'Small stream along curb', '2026-02-08T10:25:00Z'),
    (5015, 1008, 201, 'status_changed', 'open', 'resolved', 'Gasket replaced and area checked', '2026-02-09T09:05:00Z'),
    (5016, 1009, 101, 'created', NULL, 'new', 'No sharp edge visible', '2026-02-09T22:15:00Z'),
    (5017, 1010, 102, 'created', NULL, 'open', 'Current bin lid is missing', '2026-02-10T13:50:00Z'),
    (5018, 1010, 202, 'status_changed', 'open', 'closed', 'Replacement delivered', '2026-02-12T16:30:00Z'),
    (5019, 1011, 103, 'created', NULL, 'open', 'Wait exceeds one full light cycle', '2026-02-11T08:35:00Z'),
    (5020, 1012, 104, 'created', NULL, 'open', 'Possible photocell issue', '2026-02-12T14:45:00Z'),
    (5021, 1012, 202, 'status_changed', 'open', 'resolved', 'Photocell cleaned and tested', '2026-02-14T18:10:00Z');

-- Verification: these counts should be 8, 12, and 21.
SELECT 'users' AS table_name, count(*) AS row_count FROM users
UNION ALL
SELECT 'tickets', count(*) FROM tickets
UNION ALL
SELECT 'ticket_events', count(*) FROM ticket_events;
"""
run_sql(setup_sql)
FIXTURE_LOADED = True

### Read a Few Rows

`metro_support` is the schema (a namespace); `tickets` is the table.
`SELECT` chooses columns, `ORDER BY` makes display order predictable,
and `LIMIT 4` displays only four rows without removing stored data.
Expect ticket 1004 to show `NULL` for its assignee.

In [ ]:
run_sql("""
SELECT ticket_id, category, priority, status, assignee_id
FROM metro_support.tickets
ORDER BY ticket_id
LIMIT 4;
""")

## Day 1: A View Is a Named Query

A normal view stores a query definition, not a frozen copy of its
results. Querying it reads current underlying data. `AS` introduces
the defining query; `t` is a short alias for `tickets`.

This is the starter from the published
[Views and Identity lab](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_03/lab_01_views_identity.md).
`IN` accepts any listed status. Expect **7 rows**, including 1004 and
1009, with columns in the order shown below.

In [ ]:
run_sql("""
CREATE OR REPLACE VIEW metro_support.active_ticket_queue AS
SELECT t.ticket_id, t.subject, t.status, t.priority, t.opened_at
FROM metro_support.tickets AS t
WHERE t.status IN ('new', 'open', 'in_progress');

SELECT * FROM metro_support.active_ticket_queue ORDER BY ticket_id;
""")

### Your View Change: Keep Unassigned Tickets

A `LEFT JOIN` keeps each left-side ticket even when no right-side user
matches. The user's columns then contain `NULL`. An inner join would
lose unassigned tickets. Match the **assignee**, not the requester.

In your workspace below, append `users.display_name` as a new final
column named `assignee_name`. Keep the original five view columns in
the same order. `CREATE OR REPLACE VIEW` allows appending columns, not
arbitrarily reordering or renaming existing ones. Do not drop the view
to work around its contract. Avoid `SELECT *` in the definition.

Limiting columns in a view is not sufficient access control when the
same user can still read the base table; grants come later.

### Worked Experiment: Rollback Does Not Reuse an Identity Number

`GENERATED BY DEFAULT AS IDENTITY` asks PostgreSQL to allocate a number
when an insert omits `note_id`. `RETURNING` displays the number that
insert received. `BEGIN` starts a transaction, `COMMIT` keeps its
changes, and `ROLLBACK` cancels its uncommitted changes.

This resets **only the disposable `change_notes` table**, not your
tickets or view. Run the entire block together. Predict which rows
survive before reading the final result.

In [ ]:
run_sql("""
DROP TABLE IF EXISTS metro_support.change_notes;
CREATE TABLE metro_support.change_notes (
    note_id bigint GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    change_name text NOT NULL,
    created_at timestamptz NOT NULL DEFAULT now()
);

BEGIN;
INSERT INTO metro_support.change_notes (change_name)
VALUES ('Created dashboard view') RETURNING note_id;
COMMIT;

BEGIN;
INSERT INTO metro_support.change_notes (change_name)
VALUES ('Cancelled change') RETURNING note_id;
ROLLBACK;

BEGIN;
INSERT INTO metro_support.change_notes (change_name)
VALUES ('Added assignee name') RETURNING note_id;
COMMIT;

SELECT note_id, change_name
FROM metro_support.change_notes ORDER BY note_id;
""")

Each fresh run of that whole experiment returns IDs **1, 2, 3** from
the inserts, but the final table contains only **1 and 3** (2 rows).
The cancelled row is gone; its already-allocated sequence number is
not returned. The gap is consistent with successful rollback, not
proof of a lost committed row. Explain this distinction in your own
words in a SQL comment. See PostgreSQL's
[sequence explanation](https://www.postgresql.org/docs/current/functions-sequence.html).

`information_schema` exposes database definitions as queryable tables.
The next checks inspect the saved view and the identity column rather
than guessing from the data alone.

In [ ]:
run_sql("""
SELECT view_definition
FROM information_schema.views
WHERE table_schema = 'metro_support' AND table_name = 'active_ticket_queue';

SELECT column_name, is_identity, identity_generation
FROM information_schema.columns
WHERE table_schema = 'metro_support' AND table_name = 'change_notes'
ORDER BY ordinal_position;
""")

### Your Work: Complete Lab 1 Here

Put your completed view, identity experiment, definition checks, and
missing-ID explanation in the string below. SQL comments start with
`--`. Copy the supplied experiment and checks; write the assignee join
yourself. The initial workspace contains comments, not an answer.

After your view change, the checks must still find **7 tickets** and
both **1004 and 1009**. Inspect `assignee_name` for those two tickets:
missing names should be `NULL`. Submit **one** file,
`week_03_views_identity.sql`. Keep the database and view for Day 2.

In [ ]:
views_identity_sql = """
-- TODO: Define active_ticket_queue with assignee_name appended last.
-- TODO: Include the identity experiment and explain the missing ID.
-- TODO: Include the information_schema checks above.

SELECT count(*) AS active_tickets FROM metro_support.active_ticket_queue;
SELECT * FROM metro_support.active_ticket_queue
WHERE ticket_id IN (1004, 1009) ORDER BY ticket_id;
"""
run_sql(views_identity_sql)

### Optional: Download Your SQL, Not the Notebook

Finish `views_identity_sql` above, including your SQL comments and checks.
This cell saves **exactly that string**, not every cell you ran and
not your query output. Review the printed text before submitting.
Set the switch to `True` when ready; it writes `week_03_views_identity.sql` in the
runtime and opens Colab's download prompt. On local Jupyter it prints
the file's location. Repeating the export replaces that same file.
Downloading is not submitting; upload the SQL file to Brightspace.

In [ ]:
EXPORT_SQL = False  # Change to True only after finishing the workspace.
if EXPORT_SQL:
    if "-- TODO" in views_identity_sql:
        raise ValueError("Replace the TODO placeholders with your own work first.")
    print(views_identity_sql)  # Preview exactly what will be in the file.
    sql_file = Path("week_03_views_identity.sql")
    sql_file.write_text(views_identity_sql.strip() + "\n", encoding="utf-8")
    if IN_COLAB:
        from google.colab import files
        files.download(str(sql_file))
    else:
        print("Saved:", sql_file.resolve())

## Day 2: Add a Field Without Inventing History

Follow the published
[Safe Migration lab](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_03/lab_02_safe_migration.md).
Complete Day 1's view change first. **Do not reload the fixture.**
If the runtime expired, create a new practice database using setup,
then run your saved Day 1 view SQL before this precheck. Do not reset
a still-running database whose work you want to keep.
`source_channel` will record `web`, `phone`, `mobile`, or `unknown`.
There is no trustworthy channel value for every historical ticket.
A note mentioning a mobile form is not a complete historical contract.
Do not label all old tickets `web` or infer a value for each from prose.

A **migration** changes an existing database's structure or data. A
**precheck** checks assumptions before changing it. Initially the next
first query returns **0 rows** (no such column), followed by **12
tickets** and **7 active tickets**. If the column exists, inspect your
previous attempt instead of adding it a second time.

In [ ]:
run_sql("""
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'metro_support' AND table_name = 'tickets'
  AND column_name = 'source_channel';

SELECT count(*) AS total_tickets FROM metro_support.tickets;
SELECT count(*) AS active_tickets FROM metro_support.active_ticket_queue;
""")

### Worked Example: Rehearse DDL on a Temporary Table

DDL means commands that define structure, such as `ALTER TABLE`.
PostgreSQL can roll back this `ADD COLUMN`. This small demonstration
uses a temporary table, not the graded `source_channel` migration.
`information_schema.columns` shows `message, reviewed` before rollback
and only `message` afterward. The temporary table disappears when this
call's connection closes. This differs from identity allocation above.

In [ ]:
run_sql("""
CREATE TEMP TABLE ddl_practice (message text);
BEGIN;
ALTER TABLE ddl_practice ADD COLUMN reviewed boolean;
SELECT column_name FROM information_schema.columns
WHERE table_name = 'ddl_practice' AND table_schema LIKE 'pg_temp_%'
ORDER BY ordinal_position;
ROLLBACK;
SELECT column_name FROM information_schema.columns
WHERE table_name = 'ddl_practice' AND table_schema LIKE 'pg_temp_%'
ORDER BY ordinal_position;
""")

### Your Work: Rehearse the Published Migration

Finish the three TODO lines in the starter below, inside the transaction.

1. Add a **named CHECK** allowing only `web`, `phone`, `mobile`, `unknown`.
2. Set the column **NOT NULL** after backfilling the existing rows.
3. Set **DEFAULT 'unknown'** for later inserts that omit the column.

These have different jobs. A CHECK restricts allowed values, NOT NULL
rejects missing values, and DEFAULT supplies an omitted value. A default
does not replace an explicit NULL and is not a history backfill. Use
the course constraint examples as syntax references; the finished
three statements are your work.

The grouped check should show **unknown: 12** inside the transaction.
Keep `ROLLBACK` while rehearsing. Then rerun the precheck: the column
should be absent again, with counts still 12 and 7.

In [ ]:
migration_rehearsal_sql = """
BEGIN;
ALTER TABLE metro_support.tickets ADD COLUMN source_channel text;
-- Preserve uncertainty rather than inventing historical channel values.
UPDATE metro_support.tickets SET source_channel = 'unknown'
WHERE source_channel IS NULL;

-- TODO: Add your named CHECK for the four allowed values.
-- TODO: Make source_channel NOT NULL.
-- TODO: Give source_channel DEFAULT 'unknown'.

SELECT source_channel, count(*)
FROM metro_support.tickets GROUP BY source_channel;
ROLLBACK;
"""
run_sql(migration_rehearsal_sql)

In [ ]:
run_sql("""
SELECT column_name FROM information_schema.columns
WHERE table_schema = 'metro_support' AND table_name = 'tickets'
  AND column_name = 'source_channel';
SELECT count(*) AS total_tickets FROM metro_support.tickets;
SELECT count(*) AS active_tickets FROM metro_support.active_ticket_queue;
""")

### Commit Only After the Rehearsal Works

In the workspace below, include your precheck and completed migration,
this time ending it with `COMMIT`. Then append `source_channel` as the
**last** column of `active_ticket_queue`, keeping the original columns,
Day 1's `assignee_name`, their order, and the LEFT JOIN. Do not rerun a
committed ADD COLUMN; inspect state first.

Add verification queries and a short SQL comment explaining `unknown`,
the default's benefit for an older application, and why dropping this
column later could destroy newly collected real values. A later repair
may need to preserve current data rather than reverse the migration.
Submit only `week_03_safe_migration.sql`, not a separate change plan.

In [ ]:
safe_migration_sql = """
-- TODO: Include the precheck and your completed, committed migration.
-- TODO: Append source_channel to your existing view without losing columns.
-- TODO: Include verification queries and your explanation as SQL comments.
-- TODO: Keep the deliberate invalid update commented out in this file.
"""
run_sql(safe_migration_sql)

### Check Your Committed Contract

Set the next switch to `True` **after** your committed migration and
view update exist. Expect `is_nullable = NO`, a default containing
`'unknown'`, your named CHECK, and 7 active tickets including 1004 and
1009. The valid `mobile` update is rehearsed and rolled back, so it
leaves ticket 1001's original value (`unknown` on a fresh run) intact.

In [ ]:
CHECK_MY_MIGRATION = False
if CHECK_MY_MIGRATION:
    run_sql("""
    SELECT column_name, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_schema = 'metro_support' AND table_name = 'tickets'
      AND column_name = 'source_channel';
    SELECT conname, pg_get_constraintdef(oid) AS definition
    FROM pg_constraint
    WHERE conrelid = 'metro_support.tickets'::regclass AND contype = 'c';
    SELECT count(*) AS active_tickets FROM metro_support.active_ticket_queue;
    SELECT * FROM metro_support.active_ticket_queue
    WHERE ticket_id IN (1004, 1009) ORDER BY ticket_id;

    BEGIN;
    UPDATE metro_support.tickets SET source_channel = 'mobile'
    WHERE ticket_id = 1001 RETURNING ticket_id, source_channel;
    ROLLBACK;
    SELECT ticket_id, source_channel FROM metro_support.tickets
    WHERE ticket_id = 1001;
    """)

### A Separate, Deliberate Failure

After creating your constraint, set this switch to `True` once. A
correct CHECK raises an error identifying **your constraint name** and
SQLSTATE **23514** for `fax`. A missing column or other syntax error is
not evidence that the CHECK works. If the UPDATE succeeds, your rule
is missing or wrong; the final ROLLBACK still cancels the test.

When the error occurs, `psql` stops before reaching the written ROLLBACK,
but closing this call's connection rolls the uncommitted update back.
Record the observed error as a comment in your SQL file and keep the
failing statement itself commented out in the submitted file.

In [ ]:
RUN_INVALID_CHANNEL_TEST = False  # Intentionally raises an error when enabled.
if RUN_INVALID_CHANNEL_TEST:
    run_sql("""
    BEGIN;
    UPDATE metro_support.tickets SET source_channel = 'fax'
    WHERE ticket_id = 1001;
    ROLLBACK;
    """)

### Optional: Download Your SQL, Not the Notebook

Finish `safe_migration_sql` above, including your SQL comments and checks.
This cell saves **exactly that string**, not every cell you ran and
not your query output. Review the printed text before submitting.
Set the switch to `True` when ready; it writes `week_03_safe_migration.sql` in the
runtime and opens Colab's download prompt. On local Jupyter it prints
the file's location. Repeating the export replaces that same file.
Downloading is not submitting; upload the SQL file to Brightspace.

In [ ]:
EXPORT_SQL = False  # Change to True only after finishing the workspace.
if EXPORT_SQL:
    if "-- TODO" in safe_migration_sql:
        raise ValueError("Replace the TODO placeholders with your own work first.")
    print(safe_migration_sql)  # Preview exactly what will be in the file.
    sql_file = Path("week_03_safe_migration.sql")
    sql_file.write_text(safe_migration_sql.strip() + "\n", encoding="utf-8")
    if IN_COLAB:
        from google.colab import files
        files.download(str(sql_file))
    else:
        print("Saved:", sql_file.resolve())

## Finish: Remove Only This Notebook's Practice Database

**Download your two SQL files first. Do not run this between days.**
This removes only the uniquely named database created by this run.
In Colab it also stops the service started above, so finish other
PostgreSQL work in this same runtime first. It never stops your
computer's preexisting PostgreSQL service. Saved SQL files are kept.

To start over, run cleanup, then rerun from the setup cells. If the
Python kernel restarts and loses its variables, do not guess a database
name to drop. A new run creates a different practice database. Colab
discards runtime storage eventually; a local operator can inspect and
remove an old practice database separately after confirming ownership.

In [ ]:
if "PRACTICE_DB" in globals():
    assert PRACTICE_DB == "cst4714_week03_" + run_id
    subprocess.run(
        PG_PREFIX + ["dropdb"] + PG_ARGS
        + ["--maintenance-db=postgres", "--if-exists", PRACTICE_DB],
        check=True,
    )
    print("Removed:", PRACTICE_DB)
    del PRACTICE_DB
    globals().pop("FIXTURE_LOADED", None)
if globals().get("COLAB_SERVICE_STARTED", False):
    subprocess.run(["service", "postgresql", "stop"], check=True)
    COLAB_SERVICE_STARTED = False
print("Cleanup complete. A local computer's server is left running.")

**License:** prose CC BY-NC-SA 4.0; code MIT; synthetic data CC0.